In [ ]:
# @title 📦 SEL 1: Clone Repository, Pilih Git Branch & Bootstrap CUDA 12.x
# @markdown Sel ini melakukan `git clone` / `git checkout` dari repositori GitHub sesuai `GIT_BRANCH` pilihan Anda, memasang dependensi Python, dan menyiapkan binary `llama-server` & `llama-mtmd-cli` untuk Tesla T4.

import os, sys, shutil, subprocess
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/sevunk/vlm-caption-studio.git" # @param {type:"string"}
GIT_BRANCH = "main" # @param {type:"string"}
PROJECT_DIR = Path("/content/vlm-caption-studio")
FRESH_CLONE = True # @param {type:"boolean"}
MOUNT_GOOGLE_DRIVE = False # @param {type:"boolean"}

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print(f"[!] Info mount Google Drive: {e}")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU CUDA tidak terdeteksi! Pilih menu Runtime > Change runtime type > T4 GPU.")
print(f"[+] GPU Aktif: {torch.cuda.get_device_name(0)} | CUDA: {torch.version.cuda}")

# 1. Clone atau update repositori GitHub pada branch yang dipilih
repo_url = GITHUB_REPO_URL.strip()
branch = GIT_BRANCH.strip() or "main"
if not repo_url:
    raise ValueError("GITHUB_REPO_URL wajib diisi!")

if FRESH_CLONE and PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if not PROJECT_DIR.exists():
    print(f"[+] Mengkloning {repo_url} (branch: {branch}) ke {PROJECT_DIR}...")
    subprocess.run(["git", "clone", "--branch", branch, "--single-branch", repo_url, str(PROJECT_DIR)], check=True)
else:
    print(f"[+] Memperbarui repositori di {PROJECT_DIR} pada branch '{branch}'...")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", branch], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "origin", branch], check=True)

# 2. Pasang dependensi dari requirements.txt
print("[+] Memasang paket Python (Gradio, FastAPI, timm, HuggingFace Hub, CUDA 12 Runtime)...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed",
     "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True
)

# 3. Jalankan bootstrap_vlm.py untuk mengunduh & memverifikasi llama-server & llama-mtmd-cli
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from bootstrap_vlm import bootstrap_llama_cpp
bin_info = bootstrap_llama_cpp()
print("\n✅ SETUP SELESAI! Silakan jalankan SEL 2 di bawah untuk membuka Studio.")


In [ ]:
# @title 🚀 SEL 2: Launch Vision AI Labeling, PixAI Booru Tagger & Multimodal Chat Studio
# @markdown Pilih model awal dan folder dataset, lalu jalankan sel ini untuk membuka antarmuka web penuh.

import os, sys, re, time, socket, signal, subprocess
from pathlib import Path
from IPython.display import display, HTML

PROJECT_DIR = Path("/content/vlm-caption-studio")
DEFAULT_MODEL = "Qwen3.5-9B [Uncensored Aggressive - HauhauCS]" # @param ["Qwen3.5-9B [Uncensored Aggressive - HauhauCS]", "Qwen3-VL-8B [Uncensored Abliterated - wangkanai]", "JoyCaption Beta One [Llama-3.1-8B Uncensored - fancyfeast]", "Gemma-3-12B-IT [Uncensored Abliterated - mlabonne]", "Qwen3-VL-8B [Official - Qwen]", "Qwen3.5-9B [Official - Unsloth]", "Gemma-3-12B-IT [Official - Unsloth]", "PixAI Tagger v0.9 [Fast Anime & Character/IP Booru Tagger]"]
DATASET_FOLDER = "/content/dataset" # @param {type:"string"}
PREWARM_MODEL_ON_START = True # @param {type:"boolean"}
HF_TOKEN_OPTIONAL = "" # @param {type:"string"}

if not PROJECT_DIR.exists():
    raise RuntimeError(f"Folder {PROJECT_DIR} belum ada! Jalankan SEL 1 terlebih dahulu.")

os.makedirs(DATASET_FOLDER, exist_ok=True)
APP_LOG = Path("/content/vlm_studio_gradio.log")
APP_PID = Path("/content/vlm_studio_gradio.pid")

if APP_PID.exists():
    try:
        os.kill(int(APP_PID.read_text().strip()), signal.SIGTERM)
        time.sleep(1)
    except Exception:
        pass

ui_port = 7865
for p in range(7865, 7880):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(("127.0.0.1", p)) != 0:
            ui_port = p
            break

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from bootstrap_vlm import bootstrap_llama_cpp
bin_info = bootstrap_llama_cpp()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["VLM_LLAMA_SERVER"] = bin_info["llama_server"]
env["VLM_LLAMA_MTMD_CLI"] = bin_info["llama_mtmd_cli"]
env["VLM_LLAMA_CLI"] = bin_info["llama_cli"]
env["VLM_DEFAULT_MODEL"] = DEFAULT_MODEL
env["VLM_DATASET_DIR"] = DATASET_FOLDER
env["VLM_UI_PORT"] = str(ui_port)
env["VLM_PREWARM_MODEL"] = "1" if PREWARM_MODEL_ON_START else "0"
if HF_TOKEN_OPTIONAL.strip():
    env["HF_TOKEN"] = HF_TOKEN_OPTIONAL.strip()

print(f"[+] Meluncurkan Vision AI Studio di port {ui_port} dengan model '{DEFAULT_MODEL}'...")
with open(APP_LOG, "w", encoding="utf-8", errors="replace") as lf:
    proc = subprocess.Popen(
        [sys.executable, "-u", str(PROJECT_DIR / "run_vlm_studio.py")],
        env=env, stdout=lf, stderr=subprocess.STDOUT, text=True
    )
APP_PID.write_text(str(proc.pid))

public_url = None
for _ in range(360):
    if proc.poll() is not None:
        break
    time.sleep(1)
    if APP_LOG.exists():
        txt = APP_LOG.read_text(errors="ignore")
        m = re.search(r"VLM_SHARE_URL=(https://[A-Za-z0-9.-]+\.gradio\.live)", txt)
        if m:
            public_url = m.group(1)
            break

if public_url:
    print("\n✅ VISION AI LABELING, PIXAI TAGGER & CHAT STUDIO SIAP DIGUNAKAN!")
    display(HTML(
        f'<div style="padding:14px 18px;background:#11151f;border:1px solid #38bdf8;border-radius:10px;margin-top:8px;">'
        f'<a href="{public_url}" target="_blank" style="font-size:18px;font-weight:700;color:#38bdf8;text-decoration:none;">'
        f'🚀 Buka Vision AI Studio ({public_url})</a>'
        f'<p style="color:#94a3b8;font-size:12.5px;margin-top:4px;">Folder Aktif: <code>{DATASET_FOLDER}</code> · Model: <code>{DEFAULT_MODEL}</code></p></div>'
    ))
else:
    tail = "\n".join(APP_LOG.read_text(errors="ignore").splitlines()[-100:]) if APP_LOG.exists() else "(no log)"
    raise RuntimeError("Gagal memulai Gradio Studio. Log terakhir:\n" + tail)
